<a href="https://colab.research.google.com/github/varkha-d-sharma/w-b/blob/main/colabs/intro/Intro_to_Weights_%26_Biases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wandb -qU

In [2]:
# Log in to your W&B account
import wandb
import random
import math

In [13]:
wandb.login(relogin=True)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [14]:
import io
import os
import re
import sys
import yaml
import gzip
import random
import typing as t
import collections
import click
import xml.etree.ElementTree


def _process_posts(fd_in: t.IO, fd_out_train: t.IO, fd_out_test: t.IO, target_tag: str, split: int) -> None:
    for idx, line in enumerate(fd_in):
        try:
            fd_out = fd_out_train if random.random() > split else fd_out_test
            attr = xml.etree.ElementTree.fromstring(line).attrib

            pid = attr.get("Id", "")
            label = 1 if target_tag in attr.get("Tags", "") else 0
            title = re.sub(r"\s+", " ", attr.get("Title", "")).strip()
            body = re.sub(r"\s+", " ", attr.get("Body", "")).strip()
            text = title + " " + body

            fd_out.write("{}\t{}\t{}\n".format(pid, label, text))
        except Exception as ex:
            sys.stderr.write(f"Skipping the broken line {idx}: {ex}\n")

# Run Prepare Stage
# 1️. Start a new run to track this script
run = wandb.init(
   # Set the project where this run will be logged
   project="example",
   # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
   name=f"prepare",
   # Track hyperparameters and run metadata
)

random.seed(20170428)

input_artifact = wandb.Artifact(name="input_data", type="dataset")
# not sure where to put custom properties.
input_artifact.add_file(local_path="./data.xml.gz", name="training_dataset")
run.use_artifact(input_artifact)

# Specify the name of the collection and registry
# you want to publish the artifact to
COLLECTION_NAME = "first-collection"
REGISTRY_NAME = "dataset"

# Link the artifact to the registry
run.link_artifact(
   artifact=input_artifact,
        target_path=f"wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}"
)


#Dataset = collections.namedtuple('Dataset', ['train', 'test'])
#    output_ds = Dataset(train=os.path.join(output_dir, "train.tsv"), test=os.path.join(output_dir, "test.tsv"))
with gzip.open("./data.xml.gz", "rb") as fd_in,\
    io.open("train.tsv", "w", encoding="utf8") as fd_out_train,\
    io.open("test.tsv", "w", encoding="utf8") as fd_out_test:
    _process_posts(fd_in, fd_out_train, fd_out_test, "<python>", 0.20)


op_artifact_1 = wandb.Artifact(name="train.tsv", type="dataset")
# not sure where to put custom properties.
op_artifact_1.add_file(local_path="./train.tsv", name="training_dataset")
run.log_artifact(op_artifact_1)

# Link the artifact to the registry
run.link_artifact(
   artifact=op_artifact_1,
        target_path=f"wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}"
)



op_artifact_2 = wandb.Artifact(name="test.tsv", type="dataset")
# not sure where to put custom properties.
op_artifact_2.add_file(local_path="./test.tsv", name="training_dataset")
run.log_artifact(op_artifact_2)

# Link the artifact to the registry
run.link_artifact(
   artifact=op_artifact_2,
        target_path=f"wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}"
)


# Mark the run as finished
wandb.finish()

CommError: ArtifactSaver.createArtifact: returned error 400: {"data":{"createArtifact":null},"errors":[{"message":"Invalid Client ID digest","path":["createArtifact"]}]}

In [8]:
import os
import sys
import yaml
import pickle
import click
import collections
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from sklearn.feature_extraction.text import (CountVectorizer, TfidfTransformer)


def _get_df(data: str) -> pd.DataFrame:
    df = pd.read_csv(
        data,
        encoding="utf-8",
        header=None,
        delimiter="\t",
        names=["id", "label", "text"],
    )
    sys.stderr.write(f"The input data frame {data} size is {df.shape}\n")
    return df


def _save_matrix(df: pd.DataFrame, matrix, output: str) -> None:
    id_matrix = sparse.csr_matrix(df.id.astype(np.int64)).T
    label_matrix = sparse.csr_matrix(df.label.astype(np.int64)).T

    result = sparse.hstack([id_matrix, label_matrix, matrix], format="csr")

    msg = "The output matrix {} size is {} and data type is {}\n"
    sys.stderr.write(msg.format(output, result.shape, result.dtype))

    with open(output, "wb") as fd:
        pickle.dump(result, fd)



# Run Featurize Stage
# 1️. Start a new run to track this script
run = wandb.init(
   # Set the project where this run will be logged
   project="example",
   # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
   name=f"featurize",
   # Track hyperparameters and run metadata
)



np.set_printoptions(suppress=True)

#input_artifact = wandb.Artifact(name="input_data", type="dataset")
# not sure where to put custom properties.
#input_artifact.add_file(local_path="./data.xml.gz", name="training_dataset")
run.use_artifact(op_artifact_1)


# Generate train feature matrix
df_train = _get_df("./train.tsv")
train_words = np.array(df_train.text.str.lower().values.astype("U"))

bag_of_words = CountVectorizer(
    stop_words="english", max_features=3000, ngram_range=(1, 2)
)

bag_of_words.fit(train_words)
train_words_binary_matrix = bag_of_words.transform(train_words)
tfidf = TfidfTransformer(smooth_idf=False)
tfidf.fit(train_words_binary_matrix)
train_words_tfidf_matrix = tfidf.transform(train_words_binary_matrix)

_save_matrix(df_train, train_words_tfidf_matrix, "./train.pkl")

# Generate test feature matrix
df_test = _get_df("./test.tsv")
test_words = np.array(df_test.text.str.lower().values.astype("U"))
test_words_binary_matrix = bag_of_words.transform(test_words)
test_words_tfidf_matrix = tfidf.transform(test_words_binary_matrix)

_save_matrix(df_test, test_words_tfidf_matrix, "./test.pkl")

op_artifact_3 = wandb.Artifact(name="train.pkl", type="dataset")
# not sure where to put custom properties.
op_artifact_3.add_file(local_path="./train.pkl", name="training_dataset")
run.log_artifact(op_artifact_3)

# Link the artifact to the registry
run.link_artifact(
   artifact=op_artifact_3,
        target_path=f"wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}"
)



op_artifact_4 = wandb.Artifact(name="test.pkl", type="dataset")
# not sure where to put custom properties.
op_artifact_4.add_file(local_path="./test.pkl", name="training_dataset")
run.log_artifact(op_artifact_4)

# Link the artifact to the registry
run.link_artifact(
   artifact=op_artifact_4,
        target_path=f"wandb-registry-{REGISTRY_NAME}/{COLLECTION_NAME}"
)



# Mark the run as finished
wandb.finish()

ValueError: ArtifactSaver.createArtifact: returned error 400: {"data":{"createArtifact":null},"errors":[{"message":"Invalid Client ID digest","path":["createArtifact"]}]}